# I2C device initialization

In [ ]:
from time import sleep as delay
from project.sc8563 import project

if "ic" in globals():
    ic.close()
    
ic = project(device="sc8563", revision="1p1", emulator="ch341", logging=False, is_gui=False)

# Equipments initialization

In [ ]:
from phy.multimeter import keithley_dm6500
from phy.power_supply import rigol_dp821a, rigol_dp811a
from phy.scope import tektronix_mdo34

# from phy.eloader import sdl1030x
# from phy.relay_16ch import relay_box

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np
from time import sleep as delay
chart = plot()

dm = keithley_dm6500()
ps = rigol_dp821a()
ds = tektronix_mdo34()
# sm = keithley_2470()
# ld = sdl1030x()

# relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)

# --------------------------------------------------
# list_voltage = list(np.arange(5, 8, 0.005))
# voltage  = [round(num, 3) for num in list_voltage]
# --------------------------------------------------

# Test code block

In [ ]:
# --------------------------------------------------
list_temp = list(np.arange(0.1, 5.01, 0.1))
list_voltage  = [round(num, 2) for num in list_temp]
# --------------------------------------------------

ps.ch1.cfg_all = 0.1, 0.1
ps.ch1.power_recycle

for voltage in list_voltage:
    ps.ch1.vset = voltage
    vdd = dm.voltage_10
    try:
        reg_00h = ic.read_byte(0)
    except:
        reg_00h = "NACK"
    print(f"vout={voltage:.1f}, vdd={vdd:.03f}, reg_00h={reg_00h}")

ps.ch1.cfg_all = 0.1, 0.1
ps.ch1.disable

In [ ]:
from interface.cui_logger import logger as log
from tabulate import tabulate as tb
from time import sleep as delay
import crcmod, serial, sys, os


In [ ]:
log.scan_uart()

In [ ]:
dev = serial.Serial(port="COM7", timeout=3, baudrate=115200)

In [ ]:
packet = ["01", "11", "04", "03", "01", "00", "00"]

In [ ]:
def crc_generate(value):
    
    crc16 = crcmod.mkCrcFun(
        0x18005,
        rev=True,
        initCrc=0xFFFF,
        xorOut=0x0000
        )
    byte_array   = bytes.fromhex(value)
    crc_checksum = crc16(byte_array)

    second_crc = str(f"{crc_checksum:04X}")[0:2]
    first_crc = str(f"{crc_checksum:04X}")[2:4]
    print(f"crc={crc16}, byte array={byte_array}, crc checksum={crc_checksum}, 2nd crc={second_crc}, first crc={first_crc}")
    
    return [first_crc, second_crc]

In [ ]:
crc = crc_generate(" ".join(packet))
print(crc)

In [ ]:
packet.extend(crc)
print(packet)

In [ ]:
def send_packet(packet):
    
    for c in packet:
        dev.write(bytes.fromhex(c))
        print(f"packet={packet}, send packet={bytes.fromhex(c)}")

def read_packet(packet):

    send_packet(packet)

    ret = dev.readline()
    hex_list = [f'{byte:#04x}' for byte in ret]
    print(f"read packet={hex_list}")
    
    return ret, hex_list

In [ ]:
send_packet(packet=packet)